# Introduction to Building APIs in Python

In this lesson, we'll learn what an API is, why it's useful, and how to create a simple RESTful API with Flask. We'll cover:
- Installing Flask
- Creating a minimal Flask app
- Handling routes (GET, POST, etc.)
- Returning JSON data
- Testing the API

**Note**: Running a Flask app in the same notebook can be tricky, but we'll outline the steps here. In practice, you often keep API code in a separate file (e.g., `app.py`) and run it with `python app.py`.

## 1. What is an API?

- **API (Application Programming Interface)**: A way for software applications to communicate with each other.
- **RESTful API**: A style of API design that typically uses HTTP methods (GET, POST, PUT, DELETE, etc.) and resource-based endpoints.
- **Flask**: A lightweight web framework for Python, useful for building APIs quickly.

## 2. Installing Flask

Open a terminal (or Anaconda prompt) and install Flask:

```bash
pip install flask
```


In [ ]:

# After installation, you can import Flask directly in Python.
# Let's check the Flask version
import importlib
import flask
print("Flask version:", importlib.metadata.version("flask"))


## 3. Creating a Minimal Flask App

Below is a *very basic* Flask application that listens for HTTP requests and returns a \"Hello, World!\" message. Typically, you'd save this in a file called `app.py`, then run `python app.py`.

For this notebook, we'll just show the code. 


In [ ]:
# app.py (example code)

from flask import Flask

app = Flask(__name__)

@app.route('/')
def home():
    return "Hello, World from Flask API!"

# The check below ensures the app only runs if it's the main module.
# If you copy this into a file named app.py, you'd run `python app.py` in a terminal.
if __name__ == "__main__":
    # debug=True is convenient for development because it auto-reloads on changes
    app.run(debug=True)


**Explanation**:
- `Flask(__name__)` initializes a Flask application.
- `@app.route('/')` decorates a function that defines a route at the root URL (`/`).
- When a request hits `http://127.0.0.1:5000/`, the server calls the `home()` function and returns the string.

**Running**:
- Save as `app.py`.
- Run `python app.py`.
- Go to `http://127.0.0.1:5000/` in your browser to see the message.


## 4. Returning JSON Data

APIs often return data in JSON format. Let's add another route that returns JSON describing a simple resource (e.g., a list of \"books\").


In [ ]:
# app.py
from flask import Flask, jsonify

app = Flask(__name__)

# In-memory data (for demonstration)
books = [
    {"id": 1, "title": "The Hobbit", "author": "J.R.R. Tolkien"},
    {"id": 2, "title": "1984", "author": "George Orwell"},
]

@app.route("/")
def home():
    return "Hello, World from Flask API!"

@app.route("/books", methods=["GET"])
def get_books():
    return jsonify(books)  # Convert Python list/dict to JSON

if __name__ == "__main__":
    app.run(debug=True)


**Explanation**:
- We store some sample data in a Python list of dictionaries.
- `jsonify(...)` converts Python objects into JSON, setting the correct content type (`application/json`).
- When you visit `http://127.0.0.1:5000/books`, you'll receive a JSON array of books.


## 5. Creating a POST Endpoint

Let's allow clients to create a new book by sending a JSON payload. We use the `request` object from Flask to get JSON data from the request body.


In [ ]:
# app.py
from flask import Flask, jsonify, request

app = Flask(__name__)

books = [
    {"id": 1, "title": "The Hobbit", "author": "J.R.R. Tolkien"},
    {"id": 2, "title": "1984", "author": "George Orwell"},
]

@app.route("/books", methods=["GET"])
def get_books():
    return jsonify(books)

@app.route("/books", methods=["POST"])
def create_book():
    new_book = request.get_json()  # parse JSON body
    # In reality, you'd validate the input carefully here.
    new_id = max(book["id"] for book in books) + 1
    new_book["id"] = new_id
    books.append(new_book)
    return jsonify(new_book), 201  # 201 means \"Created\"

if __name__ == "__main__":
    app.run(debug=True)


**Explanation**:
- `request.get_json()` reads the JSON payload from the incoming POST request.
- We assign a new `id` by taking the max existing ID and adding 1.
- We return the newly created book with a 201 (Created) status code.


## 6. Testing Your API

### 6.1 Using a Web Browser
For GET requests, you can often just open the URL in your browser:
- `http://127.0.0.1:5000/books`

### 6.2 Using `requests` in Python


In [ ]:
import requests

# GET request
response = requests.get("http://127.0.0.1:5000/books")
print(response.status_code)
print(response.json())

# POST request
new_book_data = {
    "title": "Brave New World",
    "author": "Aldous Huxley"
}
response = requests.post("http://127.0.0.1:5000/books", json=new_book_data)
print(response.status_code)
print(response.json())


**Explanation**:
- The first request retrieves the list of books from our Flask server.
- The second request posts a new book and prints the returned object. If successful, we expect a 201 status code.

### 6.3 Using cURL (Optional)
```bash
curl http://127.0.0.1:5000/books

curl -X POST -H \"Content-Type: application/json\" \\
     -d '{\"title\":\"Brave New World\",\"author\":\"Aldous Huxley\"}' \\
     http://127.0.0.1:5000/books
```


## 7. Adding More Endpoints
### 7.1 Retrieving a Single Book

In [ ]:
@app.route("/books/<int:book_id>", methods=["GET"])
def get_book(book_id):
    for book in books:
        if book["id"] == book_id:
            return jsonify(book)
    return jsonify({"error": "Book not found"}), 404

### 7.2 Deleting a Book

In [ ]:
@app.route("/books/<int:book_id>", methods=["DELETE"])
def delete_book(book_id):
    for i, book in enumerate(books):
        if book["id"] == book_id:
            books.pop(i)
            return jsonify({}), 204  # 204 = \"No Content\"
    return jsonify({"error": "Book not found"}), 404


**Explanation**:
- `<int:book_id>` is a path parameter that captures a book ID as an integer.
- We search the `books` list for a matching ID, then return or delete as needed.


## 8. Common Best Practices

1. **Use proper HTTP verbs**: GET for retrieving, POST for creating, PUT/PATCH for updating, DELETE for deleting.
2. **Return appropriate status codes**: 200 for success, 201 for creation, 404 for not found, 400 for invalid data, etc.
3. **Validate input**: Before adding or updating data, ensure the request body is valid.
4. **Use a real database**: For production, integrate a database (e.g., SQLite, PostgreSQL) instead of in-memory lists.
5. **Document your API**: Provide information about endpoints, parameters, request/response formats.
6. **Keep your code modular**: Split routes, models, and configuration into different files as your project grows.
